Ten skrypt ładuje dane do schematu hurtowni danych

In [0]:
%sql
MERGE INTO dw.dim_car AS Dim_car
USING staging.car AS src
ON Dim_car.car_inventory_id = src.car_inventory_id
WHEN MATCHED AND (Dim_car.valid_from < src.last_update) THEN 
  UPDATE SET
    is_active = false,
    valid_to = src.last_update,
    dw_update_date = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (car_inventory_id, car_id, car_producer, car_model, car_production_year, car_fuel_type, car_license_plates, car_current_rental_rate, car_purchase_price, car_sell_price, valid_from, valid_to, is_active, dw_insert_date)
VALUES(src.car_inventory_id, src.car_id, src.car_producer, src.car_model, src.car_production_year, src.car_fuel_type, src.car_license_plates, src.car_current_rental_rate, src.car_purchase_price, src.car_sell_price, src.last_update, '9999-12-31',TRUE,CURRENT_TIMESTAMP())


In [0]:
%sql 

MERGE INTO dw.dim_customer d 
USING landing.customer l 
ON  l.customer_id = d.customer_id 
WHEN MATCHED AND d.is_active = True AND d.valid_from < l.last_update THEN  
  UPDATE SET  
    d.is_active = False, 
    d.dw_update_date = current_timestamp(), 
    d.valid_to = current_timestamp() 

WHEN NOT MATCHED BY SOURCE AND d.is_active = True THEN 
  UPDATE SET  
    d.is_active = False, 
    d.dw_update_date = current_timestamp(), 
    d.valid_to = current_timestamp() 


In [0]:
%sql 

INSERT INTO dw.dim_customer (customer_id,customer_first_name,customer_last_name,customer_email,customer_birth_date,customer_address,customer_city,customer_country,customer_postal_code,dw_insert_date,valid_from,valid_to,is_active)  
  SELECT c.customer_id,c.first_name,c.last_name,c.email,c.birth_date,coalesce(a.address,'NA'),coalesce(a.city_id,'NA'),coalesce(country,'NA'),  
  postal_code,CURRENT_TIMESTAMP(),c.last_update,'9999-12-31' AS valid_to,TRUE  
  FROM landing.customer c  
  LEFT JOIN dw.dim_customer dw ON c.customer_id=dw.customer_id AND dw.is_active=TRUE 
  LEFT JOIN landing.address a ON c.address_id=a.address_id  
  LEFT JOIN landing.city ON a.city_id=city.city_id  
  LEFT JOIN landing.country ON city.country_id=country.country_id  
  WHERE dw.customer_id IS NULL; 

In [0]:
%sql
INSERT INTO dw.fact_service (service_id, car_key, service_date_key, service_type, service_cost)
SELECT 
    Service.service_id,
    Dim_car.car_key,
    Dim_service_date.service_date_key,
    Service.service_type,
    Service.service_cost
FROM landing.service Service
INNER JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Service.inventory_id
INNER JOIN dw.dim_service_date Dim_service_date ON Dim_service_date.service_date = Service.service_date
LEFT JOIN dw.fact_service Fact_service ON Fact_service.service_id = Service.service_id
WHERE Fact_service.service_id IS NULL;

In [0]:
%sql
--do naprawy

-- INSERT INTO dw.fact_rental (rental_id, customer_key, car_key, store_key, staff_key, eqipment_group_key, rental_date_key, return_date, payment_date, payment_deadline, payment_delay_days, rental_amount, payment_amount, daily_rental_rate, rental_duration)
-- SELECT
--     Rental.rental_id,
--     Dim_customer.customer_key,
--     Dim_car.car_key,
--     Dim_store.store_key,
--     Dim_staff.staff_key,
--     Dim_equipment_group.equipment_group_key,
--     Dim_rental_date.rental_date_key,
--     Rental.return_date,
--     Rental.payment_date,
--     Rental.payment_deadline,
--     DATEDIFF(Rental.payment_date, Rental.rental_date) AS payment_delay_days,
--     Rental.rental_amount,
--     Rental.payment_amount,
--     Rental.daily_rental_rate,
--     DATEDIFF(Rental.return_date, Rental.rental_date) AS rental_duration
-- FROM landing.rental Rental
-- INNER JOIN dw.dim_customer Dim_customer ON Dim_customer.customer_id = Rental.customer_id
-- INNER JOIN dw.dim_car Dim_car ON Dim_car.car_id = Rental.car_id
-- INNER JOIN dw.dim_store Dim_store ON Dim_store.store_id = Rental.store_id
-- INNER JOIN dw.dim_staff Dim_staff ON Dim_staff.staff_id = Rental.staff_id
-- INNER JOIN dw.dim_equipment_group Dim_equipment_group ON Dim_equipment_group.equipment_group_key = Rental.equipment_group_key
-- INNER JOIN dw.dim_rental_date Dim_rental_date ON Dim_rental_date.rental_date = Rental.rental_date
-- LEFT JOIN dw.fact_rental Fact_rental ON Fact_rental.rental_id = Rental.rental_id
-- WHERE Fact_rental.rental_id IS NULL

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_staff_change AS
SELECT 1 AS changed
FROM (
    SELECT staff_id, manager_id FROM landing.staff
    EXCEPT
    SELECT staff_id, staff_manager_id FROM dw.dim_staff
) diff
LIMIT 1;

In [0]:
%sql
DELETE FROM dw.bridge_staff_hierarchy
WHERE (SELECT COUNT(*) FROM tmp_staff_change) > 0;

In [0]:
%sql
WITH RECURSIVE emp_hierarchy AS (
    SELECT
        e.staff_id,
        e.manager_id,
        1 AS distance
    FROM landing.staff e
    WHERE e.manager_id IS NOT NULL
      AND e.manager_id != e.staff_id
    UNION ALL
    SELECT
        eh.staff_id,
        m.manager_id,
        eh.distance + 1 AS distance
    FROM emp_hierarchy eh
    JOIN landing.staff m
        ON eh.manager_id = m.staff_id
    WHERE m.manager_id IS NOT NULL
      AND m.manager_id != m.staff_id
),
is_empty AS (
    SELECT 1 AS empty
    FROM dw.bridge_staff_hierarchy
    LIMIT 1
)
INSERT INTO dw.bridge_staff_hierarchy(employee_id, manager_id, distance, dw_insert_date)
SELECT staff_id, manager_id, distance, current_timestamp()
FROM emp_hierarchy
WHERE
    (SELECT COUNT(*) FROM is_empty) = 0
    OR (SELECT COUNT(*) FROM tmp_staff_change) > 0;

In [0]:
%sql
MERGE INTO dw.dim_equipment d
USING landing.equipment l
ON d.equipment_id = l.equipment_id 
WHEN MATCHED AND md5(CONCAT(d.equipment_name,d.equipment_type,d.equipment_version)) != md5(CONCAT(l.name,l.type,l.version)) THEN UPDATE SET 
  d.equipment_name = l.name,
  d.equipment_type = l.type,
  d.equipment_version = l.version,
  d.dw_update_date = current_timestamp()
WHEN NOT MATCHED THEN INSERT (equipment_id,equipment_name,equipment_type,equipment_version)
VALUES(l.equipment_id,l.name,l.type,l.version)
